In [1]:
import pandas as pd
import sqlite3

# Carregando o arquivo limpo
df = pd.read_csv('../data/processed/olist_limpo.csv')

# Criando o banco de dados SQLite na memória
conn = sqlite3.connect(':memory:')

# Salvando o dataframe como tabela SQL
df.to_sql('olist', conn, index=False, if_exists='replace')

print(f"Banco criado com sucesso! Tabela olist: {df.shape}")

Banco criado com sucesso! Tabela olist: (117601, 33)


In [2]:
# Função para rodar queries SQL e mostrar resultado
def sql(query):
    return pd.read_sql_query(query, conn)

## Pergunta 1 — Quais os 10 produtos mais vendidos?

In [3]:
sql("""
SELECT 
    product_category_name,
    COUNT(*) as total_vendas
FROM olist
WHERE order_status = 'delivered'
GROUP BY product_category_name
ORDER BY total_vendas DESC
LIMIT 10
""")

,product_category_name,total_vendas
0,cama_mesa_banho,11650
1,beleza_saude,9759
2,esporte_lazer,8733
3,moveis_decoracao,8557
4,informatica_acessorios,7898
5,utilidades_domesticas,7172
6,relogios_presentes,6065
7,telefonia,4603
8,ferramentas_jardim,4464
9,automotivo,4284


## Pergunta 2 — Qual cliente mais gastou?

In [4]:
sql("""
SELECT 
    customer_id,
    ROUND(SUM(payment_value), 2) as total_gasto
FROM olist
WHERE order_status = 'delivered'
GROUP BY customer_id
ORDER BY total_gasto DESC
LIMIT 10
""")

,customer_id,total_gasto
0,1617b1357756262bfa56ab541c47bc16,109312.64
1,bd5d39761aa56689a265d95d8d32b8be,45256.00
2,be1b70680b9f9694d8c70f41fa3dc92b,44048.00
3,05455dfa7cd02f13d132aa7a6a9729c6,36489.24
4,1ff773612ab8934db89fd5afa8afe506,30186.00
5,ec5b2ba62e574342386871631fafd3fc,29099.52
6,e7d6802668de6e74d0d6c56565bf2a24,22346.60
7,8c20d9bfbc96c5d39025d77a3ba83d7f,21874.05
8,f7622098214b4634b7fe7eee269b5426,19457.04
9,71901689c5f3e5adc27b1dd16b33f0b8,19174.38


## Pergunta 3 — Qual mês teve o maior faturamento?

In [5]:
sql("""
SELECT 
    strftime('%Y-%m', order_purchase_timestamp) as ano_mes,
    ROUND(SUM(payment_value), 2) as faturamento
FROM olist
WHERE order_status = 'delivered'
GROUP BY ano_mes
ORDER BY faturamento DESC
LIMIT 10
""")

,ano_mes,faturamento
0,2017-11,1548682.69
1,2018-05,1480667.59
2,2018-04,1466607.15
3,2018-03,1435458.33
4,2018-01,1374064.02
5,2018-07,1307228.18
6,2018-06,1285926.11
7,2018-02,1279970.45
8,2018-08,1211240.09
9,2017-12,1020067.26


## Pergunta 4 — Qual vendedor tem maior ticket médio?

In [6]:
sql("""
SELECT 
    seller_id,
    COUNT(DISTINCT order_id) as total_pedidos,
    ROUND(SUM(payment_value) / COUNT(DISTINCT order_id), 2) as ticket_medio
FROM olist
WHERE order_status = 'delivered'
GROUP BY seller_id
HAVING total_pedidos >= 10
ORDER BY ticket_medio DESC
LIMIT 10
""")

,seller_id,total_pedidos,ticket_medio
0,f08a5b9dd6767129688d001acafc21e5,11,4400.13
1,59417c56835dd8e2e72f91f809cd4092,18,2092.10
2,40db9e9aa57f7bb151bcda6b0f9bdbb7,12,1985.33
3,9803a40e82e45418ab7fb84091af5231,11,1933.20
4,2bf6a2c1e71bbd29a4ad64e6d3c3629f,30,1869.17
5,b1b3948701c5c72445495bd161b83a4c,14,1566.02
6,b839e41795b7f3ad94cc2014a52f6796,33,1536.30
7,52d76513f0c4d97f3b99570e2c94ee31,19,1466.95
8,c3acdfac4e3e97ff87529454fbc03642,12,1421.44
9,04aa0a1c5ce6b222003403a3e11c3cc0,11,1316.11


## Pergunta 5 — Qual o estado com maior número de pedidos?

In [7]:
sql("""
SELECT 
    customer_state,
    COUNT(DISTINCT order_id) as total_pedidos,
    ROUND(SUM(payment_value), 2) as faturamento_total
FROM olist
WHERE order_status = 'delivered'
GROUP BY customer_state
ORDER BY total_pedidos DESC
""")

,customer_state,total_pedidos,faturamento_total
0,SP,40500,7403993.29
1,RJ,12350,2688933.90
2,MG,11354,2281229.16
3,RS,5345,1110976.47
4,PR,4923,1030822.39
5,SC,3546,767093.97
6,BA,3256,773182.02
7,DF,2080,421374.86
8,ES,1995,398321.90
9,GO,1957,493068.70


## Pergunta 6 — Qual a forma de pagamento mais utilizada?

In [10]:
sql("""
SELECT 
    payment_type,
    COUNT(*) as total_pedidos,
    ROUND(SUM(payment_value), 2) as valor_total,
    ROUND(AVG(payment_installments), 1) as media_parcelas
FROM olist
WHERE order_status = 'delivered'
GROUP BY payment_type
ORDER BY total_pedidos DESC
""")

,payment_type,total_pedidos,valor_total,media_parcelas
0,credit_card,84896,15190241.73,3.6
1,boleto,22362,3943080.78,1.0
2,voucher,6123,396110.42,1.0
3,debit_card,1654,246727.51,1.0
